In [1]:
import pandas as pd
import numpy as np
import re

# ================================
# 1. INSTALL / IMPORT
# ================================
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import wordnet
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

# ================================
# 2. LOAD DATA
# ================================
text_df = pd.read_csv('dataset/Symptom2DiseaseTextual.csv')
tab_df = pd.read_csv('dataset/BanglaTabularDataset.csv')

tab_df.columns = tab_df.columns.str.strip()

# ================================
# 3. GET SYMPTOMS
# ================================
symptoms = tab_df.columns.drop('prognosis').tolist()

# ================================
# 4. NORMALIZE FUNCTION
# ================================
def normalize(symptom):
    return symptom.lower().replace('_', ' ')

# ================================
# 5. GET SYNONYMS FROM WORDNET
# ================================
def get_wordnet_synonyms(word):

    synonyms = set()

    for syn in wordnet.synsets(word):

        for lemma in syn.lemmas():

            synonyms.add(lemma.name().replace('_', ' '))

    return list(synonyms)

# ================================
# 6. GENERATE FULL VARIANTS
# ================================
def generate_variants(symptom):

    symptom = normalize(symptom)

    words = symptom.split()

    variants = set()

    variants.add(symptom)

    for w in words:

        variants.add(w)

        syns = get_wordnet_synonyms(w)

        for s in syns:
            variants.add(s)

    stemmed_variants = set()

    for v in variants:

        stemmed = " ".join([stemmer.stem(w) for w in v.split()])

        stemmed_variants.add(stemmed)

    variants = variants.union(stemmed_variants)

    return list(variants)

# ================================
# 7. BUILD SYMPTOM MAP
# ================================
symptom_map = {}

for s in symptoms:
    symptom_map[s] = generate_variants(s)

# ================================
# 8. CLEAN TEXT
# ================================
def clean_text(text):
    return re.sub(r'[^a-zA-Z\s]', '', str(text).lower())

text_df['text_clean'] = text_df['text'].apply(clean_text)

# ================================
# 9. STEM TEXT
# ================================
def preprocess_text(text):

    words = text.split()

    return " ".join([stemmer.stem(w) for w in words])

# ================================
# 10. FEATURE EXTRACTION
# ================================
def extract_features(text, symptoms, symptom_map):

    text_stemmed = preprocess_text(text)

    features = []

    for s in symptoms:

        variants = symptom_map[s]

        if any(v in text_stemmed for v in variants):

            features.append(1)

        else:

            features.append(0)

    return features

# ================================
# 11. GENERATE FEATURE MATRIX
# ================================
X = np.array([
    extract_features(t, symptoms, symptom_map)
    for t in text_df['text_clean']
])

# ================================
# 12. CREATE FEATURE DATAFRAME
# ================================
features_df = pd.DataFrame(X, columns=symptoms)

# ================================
# 13. CREATE ENSEMBLE DATASET
# ================================
ensemble_df = pd.concat(
    [
        text_df['text'],        # original text
        features_df,            # tabular features
        text_df['label']        # label
    ],
    axis=1
)

# ================================
# 14. SAVE DATASET
# ================================
ensemble_df.to_csv('dataset/ensemble_dataset.csv', index=False)

# ================================
# 15. OUTPUT
# ================================
print(ensemble_df.head())

print("\nShape:", ensemble_df.shape)

print("\nTotal symptoms:", len(symptoms))

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mailt\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mailt\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


                                                text  Body_Pain  \
0  I have been experiencing a skin rash on my arm...          1   
1  My skin has been peeling, especially on my kne...          0   
2  I have been experiencing joint pain in my fing...          1   
3  There is a silver like dusting on my skin, esp...          0   
4  My nails have small dents or pits in them, and...          1   

   Burning_Sensation  Itching  Excessive_Hunger  Frequent_Urination  \
0                  0        1                 0                   0   
1                  1        0                 0                   0   
2                  0        0                 0                   0   
3                  0        1                 0                   0   
4                  0        0                 0                   0   

   Eye_Problems  Joint_Pain  Numbness  Weight_Gain  ...  Constipation  \
0             0           0         0            0  ...             0   
1             0         